# 🔮 Module 3 — Prédiction du Churn Client
## Stage de Fin d'Études | Marjane Maroc | Data Science

### Contexte
Ce notebook s'inscrit dans la continuité des modules précédents : après la segmentation RFM et l'analyse du panier, l'objectif est de prédire les clients susceptibles de quitter l'enseigne.

### Objectif
Construire un modèle supervisé capable d'estimer la probabilité de churn de chaque client afin d'identifier les profils à réactiver en priorité.

### Approche
- Définition d'une variable cible basée sur l'inactivité client.
- Création de variables explicatives à partir des métriques RFM.
- Entraînement et comparaison de modèles de classification.
- Export des prédictions pour Power BI et sauvegarde du modèle pour Streamlit.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, accuracy_score)
from xgboost import XGBClassifier

print("✅ Bibliothèques chargées avec succès !")
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "donnees"
POWERBI_DIR = PROJECT_DIR / "powerbi"
MODEL_DIR = PROJECT_DIR / "modeles"

## Étape 1 — Importation des bibliothèques
Chargement des bibliothèques nécessaires pour :
- manipuler les données avec `pandas` et `numpy` ;
- visualiser les résultats avec `matplotlib` et `seaborn` ;
- entraîner les modèles avec `scikit-learn` et `XGBoost` ;
- évaluer la performance avec l'accuracy, le rapport de classification et l'AUC-ROC.

In [ ]:
# Charger directement les données RFM depuis Notebook 01
rfm = pd.read_csv(DATA_DIR / 'rfm_export.csv')

print(f"✅ Données RFM chargées depuis Notebook 01")
print(f"📊 Shape    : {rfm.shape}")
print(f"🧍 Clients  : {rfm.shape[0]}")
print(f"📋 Colonnes : {list(rfm.columns)}")
print(f"\n{rfm.describe().round(2)}")

## Étape 2 — Chargement des données RFM
Le modèle de churn utilise la table RFM générée dans le Notebook 01.

Cette table contient une ligne par client avec les principales variables comportementales :

| Variable | Description |
|----------|-------------|
| `Recency` | Nombre de jours depuis le dernier achat |
| `Frequency` | Nombre d'achats ou de factures |
| `Monetary` | Montant total dépensé |
| `City` | Ville ou magasin associé au client |

In [ ]:
SEUIL_CHURN = 180

rfm['Churn'] = (rfm['Recency'] > SEUIL_CHURN).astype(int)

total   = len(rfm)
churnes = rfm['Churn'].sum()
actifs  = total - churnes

print(f"{'='*40}")
print(f"  Clients actifs  (0) : {actifs:>5} ({actifs/total*100:.1f}%)")
print(f"  Clients churnés (1) : {churnes:>5} ({churnes/total*100:.1f}%)")
print(f"  Total           : {total:>5}")
print(f"{'='*40}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pie([actifs, churnes],
            labels=['Actifs', 'Churnés'],
            autopct='%1.1f%%',
            colors=['#1D9E75', '#E24B4A'],
            wedgeprops=dict(width=0.5))
axes[0].set_title('Distribution Churn')

axes[1].hist(rfm[rfm['Churn']==0]['Recency'], bins=30,
             alpha=0.7, color='#1D9E75', label='Actifs')
axes[1].hist(rfm[rfm['Churn']==1]['Recency'], bins=30,
             alpha=0.7, color='#E24B4A', label='Churnés')
axes[1].axvline(x=SEUIL_CHURN, color='black',
                linestyle='--', label=f'Seuil {SEUIL_CHURN}j')
axes[1].set_xlabel('Récence (jours)')
axes[1].set_ylabel('Nombre de clients')
axes[1].set_title('Distribution de la Récence')
axes[1].legend()

plt.tight_layout()
plt.show()

## Étape 3 — Création de la variable cible Churn
### Principe
Un client est considéré comme churné lorsqu'il dépasse un seuil d'inactivité défini.

Dans ce projet, le seuil retenu est :

> **180 jours sans achat**

### Interprétation
- `Churn = 0` : client actif ;
- `Churn = 1` : client inactif ou à risque de départ.

In [ ]:
rfm['Montant_par_visite']   = rfm['Monetary'] / rfm['Frequency']
rfm['Recency_x_Frequency']  = rfm['Recency'] * rfm['Frequency']
rfm['Log_Montant']          = np.log1p(rfm['Monetary'])
rfm['Log_Frequency']        = np.log1p(rfm['Frequency'])

FEATURES = ['Recency', 'Frequency', 'Monetary',
            'Montant_par_visite', 'Recency_x_Frequency',
            'Log_Montant', 'Log_Frequency']

X = rfm[FEATURES]
y = rfm['Churn']

print(f"✅ Features prêtes : {FEATURES}")
print(f"📊 Shape X : {X.shape}")

## Étape 4 — Feature Engineering
### Objectif
Enrichir les variables RFM afin de donner plus d'information au modèle de classification.

### Variables créées
| Variable | Rôle |
|----------|------|
| `Montant_par_visite` | Montant moyen dépensé à chaque achat |
| `Recency_x_Frequency` | Interaction entre inactivité et fréquence |
| `Log_Montant` | Réduction de l'effet des très grands montants |
| `Log_Frequency` | Stabilisation de la distribution de fréquence |

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train : {X_train.shape[0]} clients")
print(f"Test  : {X_test.shape[0]} clients")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42,
                             class_weight='balanced', n_jobs=-1)
rf.fit(X_train_sc, y_train)
y_pred_rf  = rf.predict(X_test_sc)
y_proba_rf = rf.predict_proba(X_test_sc)[:, 1]

# XGBoost
scale_pos = (y_train==0).sum() / (y_train==1).sum()
xgb = XGBClassifier(n_estimators=200, learning_rate=0.05,
                     max_depth=5, scale_pos_weight=scale_pos,
                     random_state=42, eval_metric='logloss',
                     verbosity=0)
xgb.fit(X_train_sc, y_train)
y_pred_xgb  = xgb.predict(X_test_sc)
y_proba_xgb = xgb.predict_proba(X_test_sc)[:, 1]

print("\n✅ Modèles entraînés !")

## Étape 5 — Entraînement des modèles
Deux modèles supervisés sont entraînés et comparés :

- **Random Forest** : modèle robuste basé sur un ensemble d'arbres de décision.
- **XGBoost** : modèle performant de boosting, souvent efficace sur les données tabulaires.

Les données sont séparées en deux parties :
- **80% entraînement** ;
- **20% test**.

Une standardisation est appliquée pour conserver une échelle homogène entre les variables.

In [ ]:
for nom, y_pred, y_proba in [
    ('Random Forest', y_pred_rf,  y_proba_rf),
    ('XGBoost',       y_pred_xgb, y_proba_xgb)
]:
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    print(f"\n{'='*40}")
    print(f"  {nom}")
    print(f"  Accuracy : {acc:.3f}")
    print(f"  AUC-ROC  : {auc:.3f}")
    print(f"{'='*40}")
    print(classification_report(y_test, y_pred,
          target_names=['Actif','Churné']))

# Courbes ROC
plt.figure(figsize=(8, 5))
for nom, y_proba, color in [
    ('Random Forest', y_proba_rf,  '#1D9E75'),
    ('XGBoost',       y_proba_xgb, '#534AB7')
]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, color=color,
             label=f'{nom} (AUC={auc:.3f})', linewidth=2)

plt.plot([0,1],[0,1],'k--', alpha=0.4, label='Aléatoire')
plt.xlabel('Taux Faux Positifs')
plt.ylabel('Taux Vrais Positifs')
plt.title('Courbes ROC — Comparaison des modèles')
plt.legend()
plt.tight_layout()
plt.show()

## Étape 6 — Évaluation des performances
### Métriques utilisées
| Métrique | Rôle |
|----------|------|
| **Accuracy** | Mesure la proportion globale de bonnes prédictions |
| **Classification report** | Affiche précision, rappel et F1-score |
| **AUC-ROC** | Mesure la capacité du modèle à distinguer clients actifs et churnés |

Le modèle avec la meilleure AUC-ROC est retenu pour la suite du projet.

In [ ]:
feat_imp = pd.Series(xgb.feature_importances_,
                     index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
bars = plt.barh(feat_imp.index, feat_imp.values,
                color='#534AB7', alpha=0.85)
plt.xlabel('Importance')
plt.title('Feature Importance — XGBoost')

for bar, val in zip(bars, feat_imp.values):
    plt.text(val + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n📊 Variable la plus importante : {feat_imp.idxmax()}")

## Étape 7 — Importance des variables
Cette étape permet d'interpréter le modèle en identifiant les variables les plus influentes dans la prédiction du churn.

L'analyse de l'importance des variables aide à répondre à une question métier :

> Quels comportements clients expliquent le plus le risque de churn ?

In [ ]:
auc_rf  = roc_auc_score(y_test, y_proba_rf)
auc_xgb = roc_auc_score(y_test, y_proba_xgb)

meilleur = xgb if auc_xgb >= auc_rf else rf
meilleur_nom = 'XGBoost' if auc_xgb >= auc_rf else 'Random Forest'
print(f"✅ Meilleur modèle : {meilleur_nom}")

X_all_sc = scaler.transform(X)
rfm['Proba_Churn'] = meilleur.predict_proba(X_all_sc)[:, 1]
rfm['Churn_Predit'] = meilleur.predict(X_all_sc)
rfm['Risque'] = pd.cut(rfm['Proba_Churn'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Faible', 'Moyen', 'Élevé'])

output = DATA_DIR / 'churn_predictions.csv'
rfm.to_csv(output, index=False)

print(f"📁 Fichier exporté : churn_predictions.csv")
print(f"\n📊 Distribution des risques :")
print(rfm['Risque'].value_counts())
print(f"\n🔴 Clients à risque élevé : {(rfm['Risque']=='Élevé').sum()}")

## Étape 8 — Génération des prédictions churn
Le meilleur modèle est appliqué à l'ensemble des clients afin de produire :

- une probabilité de churn ;
- une prédiction binaire actif/churné ;
- un niveau de risque lisible par les managers.

### Niveaux de risque
| Probabilité de churn | Niveau |
|----------------------|--------|
| 0% à 30% | Faible |
| 30% à 60% | Moyen |
| 60% à 100% | Élevé |

Le fichier généré sert ensuite au dashboard et à l'analyse métier.

In [ ]:
import pandas as pd
import numpy as np

# Charger le fichier
churn = pd.read_csv(DATA_DIR / 'churn_predictions.csv')

# ── Fichier 1 : powerbi_clients.csv ──
powerbi_clients = churn[[
    'Customer ID', 'Recency', 'Frequency', 'Monetary',
    'City', 'Segment_Name', 'Cluster',
    'RFM_Score', 'Churn', 'Proba_Churn', 'Risque'
]].copy()

powerbi_clients.columns = [
    'Customer ID','Recence', 'Frequence', 'Montant',
    'Ville', 'Segment', 'Cluster',
    'RFM_Score', 'Churn', 'Proba_Churn', 'Risque'
]

path1 = POWERBI_DIR / 'powerbi_clients.csv'
powerbi_clients.to_csv(path1, index=False)
print(f"✅ powerbi_clients.csv exporté : {len(powerbi_clients)} clients")
print(f"Segments : {powerbi_clients['Segment'].value_counts().to_dict()}")

# ── Fichier 2 : powerbi_kpis.csv ──
kpis = pd.DataFrame({
    'Indicateur': [
        'Total Clients',
        'CA Total (MAD)',
        'Panier Moyen (MAD)',
        'Taux Churn (%)',
        'Clients Risque Élevé',
        'Clients Risque Moyen',
        'Clients Risque Faible'
    ],
    'Valeur': [
        len(powerbi_clients),
        round(powerbi_clients['Montant'].sum(), 2),
        round(powerbi_clients['Montant'].mean(), 2),
        round(powerbi_clients['Churn'].mean() * 100, 1),
        len(powerbi_clients[powerbi_clients['Risque']=='Élevé']),
        len(powerbi_clients[powerbi_clients['Risque']=='Moyen']),
        len(powerbi_clients[powerbi_clients['Risque']=='Faible'])
    ]
})

path2 = POWERBI_DIR / 'powerbi_kpis.csv'
kpis.to_csv(path2, index=False)
print(f"\n✅ powerbi_kpis.csv exporté !")
print(f"\n📊 KPIs :")
print(kpis.to_string(index=False))
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "donnees"
POWERBI_DIR = PROJECT_DIR / "powerbi"
MODEL_DIR = PROJECT_DIR / "modeles"

## Étape 9 — Préparation des fichiers Power BI
Cette étape transforme les prédictions du modèle en fichiers simples à exploiter dans Power BI.

Deux fichiers sont générés :

| Fichier | Utilisation |
|---------|-------------|
| `powerbi_clients.csv` | Détail des clients, segments, churn et risque |
| `powerbi_kpis.csv` | Indicateurs globaux pour les cartes KPI |

Ces exports alimentent les pages du dashboard Power BI.

In [ ]:
auc_rf  = roc_auc_score(y_test, y_proba_rf)
auc_xgb = roc_auc_score(y_test, y_proba_xgb)

meilleur = xgb if auc_xgb >= auc_rf else rf
meilleur_nom = 'XGBoost' if auc_xgb >= auc_rf else 'Random Forest'
print(f"✅ Meilleur modèle : {meilleur_nom}")

## Étape 10 — Sélection finale du modèle
Cette cellule rappelle le modèle retenu à partir de l'AUC-ROC.

Elle permet de conserver explicitement les variables `meilleur` et `meilleur_nom` avant la sauvegarde du modèle.

In [ ]:
import joblib
import json
import os

# Dossier de sauvegarde
model_dir = MODEL_DIR
os.makedirs(model_dir, exist_ok=True)

# Sauvegarde du meilleur modèle
joblib.dump(meilleur, os.path.join(model_dir, "churn_model.pkl"))

# Sauvegarde du scaler
joblib.dump(scaler, os.path.join(model_dir, "scaler.pkl"))

# Sauvegarde des colonnes utilisées par le modèle
with open(os.path.join(model_dir, "features.json"), "w", encoding="utf-8") as f:
    json.dump(FEATURES, f, ensure_ascii=False, indent=4)

# Sauvegarde du nom du modèle choisi
with open(os.path.join(model_dir, "model_info.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "meilleur_modele": meilleur_nom,
            "seuil_churn_jours": SEUIL_CHURN,
            "features": FEATURES
        },
        f,
        ensure_ascii=False,
        indent=4
    )

print("✅ Modèle, scaler et features sauvegardés avec succès.")
print(f"📁 Dossier : {model_dir}")